# Story Analyst Blueprint Analyzer

Thống kê blueprint và tính điểm chất lượng qua các lần run.

In [12]:
import json
from pathlib import Path

BLUEPRINT_FILE = 'D:/record_by_me/README/CiB/outputs/story_blueprint.json'

with open(BLUEPRINT_FILE, 'r', encoding='utf-8') as f:
    blueprint = json.load(f)

In [13]:
stats = {}
director = blueprint.get('director_view', {})

stats['main_characters'] = len(director.get('main_characters', []))
stats['main_conflicts'] = len(director.get('main_conflicts', []))
stats['critical_path_summary'] = len(director.get('critical_path_summary', []))
stats['top_hooks'] = len(director.get('top_hooks', []))

story_tree = blueprint.get('story_tree', {})
act_count = sequence_count = scene_count = beat_count = summary_count = 0

def traverse(node):
    global act_count, sequence_count, scene_count, beat_count, summary_count
    if node.get('summary'): summary_count += 1
    t = node.get('type')
    if t == 'act': act_count += 1
    elif t == 'sequence': sequence_count += 1
    elif t == 'scene':
        scene_count += 1
        for beat in node.get('beats', []):
            beat_count += 1
            if beat.get('summary'): summary_count += 1
    for child in node.get('children', []):
        traverse(child)

traverse(story_tree)

stats['acts'] = act_count
stats['sequences'] = sequence_count
stats['scenes'] = scene_count
stats['beats'] = beat_count
stats['summaries'] = summary_count

stats['causality_nodes'] = len(blueprint.get('causality_graph', {}).get('nodes', []))
stats['causality_edges'] = len(blueprint.get('causality_graph', {}).get('edges', []))
stats['relationship_nodes'] = len(blueprint.get('character_relationship_graph', {}).get('nodes', []))
stats['relationship_edges'] = len(blueprint.get('character_relationship_graph', {}).get('edges', []))
stats['props'] = len(blueprint.get('asset_and_prop_graph', {}).get('nodes', []))
stats['prop_states'] = len(blueprint.get('asset_and_prop_graph', {}).get('states', []))
stats['presence_rows'] = len(blueprint.get('presence_matrix', []))

for k,v in stats.items():
    print(f'{k:30}: {v}')

main_characters               : 1
main_conflicts                : 1
critical_path_summary         : 3
top_hooks                     : 3
acts                          : 2
sequences                     : 2
scenes                        : 3
beats                         : 60
summaries                     : 68
causality_nodes               : 60
causality_edges               : 60
relationship_nodes            : 5
relationship_edges            : 3
props                         : 4
prop_states                   : 240
presence_rows                 : 3


In [14]:
# Blueprint Quality Score

def clamp(x):
    return max(0, min(10, x))

director = blueprint.get('director_view', {})

completeness = (
    (1 if director.get('main_characters') else 0) +
    (1 if director.get('main_conflicts') else 0) +
    (1 if director.get('critical_path_summary') else 0) +
    (1 if director.get('top_hooks') else 0)
) / 4 * 10

cg = blueprint.get('causality_graph', {})
nodes = len(cg.get('nodes', []))
edges = len(cg.get('edges', []))
graph_density = clamp((edges / max(nodes,1)) * 10)

compression = blueprint.get('narrative_compression_model', {})
tiers = compression.get('importance_tiers', {})
compression_score = clamp(
    (sum(len(v) for v in tiers.values()) / max(stats['scenes'],1)) * 10
)

hook_score = clamp(len(director.get('top_hooks', [])) * 3)

reflection = blueprint.get('reflection_verification_rules', [])
continuity_score = clamp(len(reflection) / max(stats['scenes'],1) * 10)

overall = round((
    completeness +
    graph_density +
    compression_score +
    hook_score +
    continuity_score
) / 5, 2)

print('Completeness Score :', round(completeness,2))
print('Graph Density Score:', round(graph_density,2))
print('Compression Score :', round(compression_score,2))
print('Hook Quality Proxy:', round(hook_score,2))
print('Continuity Score :', round(continuity_score,2))
print('\nOverall Blueprint Score:', overall, '/ 10')

Completeness Score : 10.0
Graph Density Score: 10
Compression Score : 10
Hook Quality Proxy: 9
Continuity Score : 10

Overall Blueprint Score: 9.8 / 10
